# Step 4 — Relation Extraction (Structural + LLM)

Generates **all 7 relation CSVs** for the Knowledge Graph.

**Structural relations** (deterministic from Step 2 entity CSVs):
- `S_contains.csv` — guideline → disease → recommendation links
- `S_disease_phenotype.csv` — disease ↔ phenotype associations
- `S_disease_stage.csv` — disease ↔ stage associations

**LLM-derived relations** (extracted from text passages via **Llama 3.1** on GPU):
- `S_treats.csv` — drug/therapy → disease treatment relations
- `S_drug_adverse_event.csv` — drug → adverse event relations
- `S_disease_assessment.csv` — disease ↔ assessment relations
- `S_disease_cause.csv` — disease ↔ cause relations

**Prerequisites:**
1. Upload thesis repo (ZIP) or `git clone` so `pipeline/` exists under `/content`.
2. Step 1 must have produced `text_blocks.json` under `outputs/<run>/step1/`.
3. Step 2 must have produced entity CSVs under `outputs/<run>/step2/`.
4. Set `HF_TOKEN` (Colab Secrets or paste) for gated **Llama 3.1** on the Hub.
5. Use **GPU runtime** (T4 minimum, A100 recommended).

**Output:** `outputs/<run>/step4/` with 7 relation CSVs + `relation_report.json`.

In [ ]:
# Cell 1 — Install dependencies
import time as _t; _t0 = _t.time()
!pip install -q torch transformers accelerate sentencepiece
print(f"\n--- Install done in {_t.time()-_t0:.0f}s ---")

In [ ]:
# Cell 2 — Setup: find repo root, chdir
import os, sys, zipfile, time
from pathlib import Path

CONTENT = Path("/content")

# Optional override: os.environ["COLAB_THESIS_ROOT"] = "/content/MyFolder"


def find_repo_root() -> Path | None:
    env = os.environ.get("COLAB_THESIS_ROOT", "").strip()
    if env:
        p = Path(env).expanduser().resolve()
        if (p / "pipeline").is_dir():
            return p
    if (CONTENT / "pipeline").is_dir():
        return CONTENT.resolve()
    for c in sorted(CONTENT.iterdir(), key=lambda x: x.name.lower()):
        if c.is_dir() and (c / "pipeline").is_dir():
            return c.resolve()
    return None


UPLOAD_ZIP = True  # set False if you already cloned/unzipped

if UPLOAD_ZIP:
    from google.colab import files
    print("Upload thesis ZIP (root must contain pipeline/)")
    uploaded = files.upload()
    for name, data in uploaded.items():
        p = CONTENT / name
        p.write_bytes(data)
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(p, "r") as zf:
                zf.extractall(str(CONTENT))
        break

ROOT = find_repo_root()
if ROOT is None:
    raise FileNotFoundError(
        "No folder under /content contains pipeline/. "
        "Unzip at /content or set COLAB_THESIS_ROOT."
    )

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)
print("--- Setup done ---")

In [ ]:
# Cell 3 — Hugging Face token for Llama 3.1 (gated model)
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    HF_TOKEN = input('Paste HF_TOKEN (read access): ').strip()
os.environ['HF_TOKEN'] = HF_TOKEN
print('HF_TOKEN set:', bool(HF_TOKEN))
print("--- Token ready ---")

In [ ]:
# Cell 4 — Load entity index from Step 2 CSVs
import time
from pathlib import Path
from pipeline.step4.extract_relations import load_entity_index

RUN_DIR = ROOT / "outputs" / "pipeline-output18"
STEP2_DIR = RUN_DIR / "step2"
OUT_DIR = RUN_DIR / "step4"

t0 = time.time()
print(f"Loading entities from: {STEP2_DIR}")
entity_index = load_entity_index(STEP2_DIR)

# Count by type
type_counts = {}
for ent in entity_index.values():
    type_counts[ent.entity_type] = type_counts.get(ent.entity_type, 0) + 1

print(f"Loaded {len(entity_index)} entities in {time.time()-t0:.1f}s:")
for t, c in sorted(type_counts.items()):
    print(f"  {t}: {c}")
print("--- Entity index ready ---")

In [ ]:
# Cell 5 — Find relation-rich passages from text_blocks.json
import json, time
from pipeline.step4.extract_relations import find_relation_passages

t0 = time.time()
text_blocks_path = RUN_DIR / "step1" / "text_blocks.json"
text_blocks = json.loads(text_blocks_path.read_text(encoding="utf-8"))
print(f"Loaded {len(text_blocks)} text blocks (pages)")

passages = find_relation_passages(text_blocks, entity_index, min_entities=2)
print(f"Found {len(passages)} passages with 2+ entities in {time.time()-t0:.1f}s")

# Show samples
for i, p in enumerate(passages[:3]):
    ents = ', '.join(f'{e.name} ({e.entity_type})' for _, e in p.entities_found)
    print(f"\n--- Sample {i+1} (page {p.page}) ---")
    print(f"  Entities: {ents}")
    print(f"  Text: {p.text[:200]}...")
print("\n--- Passage filtering done ---")

In [ ]:
# Cell 6 — Run LLM relation extraction (GPU-heavy)
import time
from pipeline.step4.extract_relations import extract_relations_llm

HF_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
BATCH_SIZE = 4

t0 = time.time()
print(f"Starting LLM extraction: {len(passages)} passages, model={HF_MODEL}")
print(f"Batch size: {BATCH_SIZE}")
print("="*60)

relations = extract_relations_llm(
    passages, entity_index,
    hf_model=HF_MODEL,
    hf_token=HF_TOKEN,
    batch_size=BATCH_SIZE,
)

elapsed = time.time() - t0
print(f"\nExtracted {len(relations)} raw relations in {elapsed:.0f}s")

# Count by relation type
rel_counts = {}
for r in relations:
    rel_counts[r.relation] = rel_counts.get(r.relation, 0) + 1
for rtype, cnt in sorted(rel_counts.items()):
    print(f"  {rtype}: {cnt}")
print("--- LLM extraction done ---")

In [ ]:
# Cell 7 — Write all 7 relation CSVs + report
import time, json
from pipeline.step4.extract_relations import write_relation_csvs, _save_json

t0 = time.time()
OUT_DIR.mkdir(parents=True, exist_ok=True)

stats = write_relation_csvs(relations, entity_index, OUT_DIR, STEP2_DIR)

print(f"Output CSVs written to: {OUT_DIR}")
print("\nStructural relations (from entity CSVs):")
for name in ("S_contains", "S_disease_phenotype", "S_disease_stage"):
    print(f"  {name}: {stats.get(name, 0)} rows")
print("\nLLM-derived relations (from text passages):")
for name in ("S_treats", "S_drug_adverse_event", "S_disease_assessment", "S_disease_cause"):
    print(f"  {name}: {stats.get(name, 0)} rows")

report = {
    "status": "ok",
    "step2_dir": str(STEP2_DIR),
    "out_dir": str(OUT_DIR),
    "model": HF_MODEL,
    "total_pages": len(text_blocks),
    "passages_with_entities": len(passages),
    "raw_relations_extracted": len(relations),
    "csv_stats": stats,
    "elapsed_seconds": round(time.time()-t0, 1),
}
_save_json(report, OUT_DIR / "relation_report.json")
print(f"\nReport saved: {OUT_DIR / 'relation_report.json'}")
print("--- Writing done ---")

In [ ]:
# Cell 8 — Download step4 outputs to your computer (browser)
import zipfile
from pathlib import Path

_zip_name = "step4_outputs.zip"
_zip_path = Path("/tmp") / _zip_name

if not OUT_DIR.is_dir() or not any(OUT_DIR.iterdir()):
    print(f"Nothing to zip — folder empty or missing: {OUT_DIR}")
else:
    n = 0
    with zipfile.ZipFile(_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(OUT_DIR.iterdir()):
            if f.is_file():
                zf.write(f, arcname=f"step4/{f.name}")
                n += 1
    print(f"Packed {n} files from {OUT_DIR} into {_zip_path}")

    try:
        from google.colab import files
        files.download(str(_zip_path))
        print(f"--- Browser download started: {_zip_name} (contains step4/*.csv + relation_report.json) ---")
    except ImportError:
        print("Not in Google Colab — copy manually from:", OUT_DIR)

In [ ]:
# Cell 9 — Quick verification: display output CSVs
import csv
from pathlib import Path

print("=" * 60)
print("STEP 4 OUTPUT VERIFICATION")
print("=" * 60)

for csv_file in sorted(OUT_DIR.glob("S_*.csv")):
    with csv_file.open(encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    print(f"\n--- {csv_file.name} ({len(rows)} rows) ---")
    for row in rows[:5]:
        print(f"  {dict(row)}")
    if len(rows) > 5:
        print(f"  ... and {len(rows)-5} more rows")

# Show report
report_path = OUT_DIR / "relation_report.json"
if report_path.exists():
    import json
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print(f"\n--- relation_report.json ---")
    print(json.dumps(report, indent=2))

print("\n--- Step 4 complete ---")